> **用途说明（Demo Only）**
> 
> 本 notebook 仅用于演示和历史回放，不是正式运营入口。
> 
> 日常操作请使用 CLI：
> ```bash
> python -m pipeline.cli signal --date YYYY-MM-DD
> python -m pipeline.cli rebalance --date YYYY-MM-DD
> python -m pipeline.cli positions
> python -m pipeline.cli performance
> python -m pipeline.cli weekly-report --week YYYY-Www
> ```
> 
> 详见 `live/README.md`。

# 13 — 模拟盘演示 & 2025 全年回放

> **目的**：验证 `live/` 模块可用性，用 PaperTrader 回放 2025 全年，评估策略在真实时间序列下的绩效。

## 前提条件
- Phase 4 多因子策略已完成（`strategies/multi_factor.py`）
- 本地 CSV 数据覆盖 2025 年（`utils/local_data_loader.py`）
- `live/paper_trader.py` 和 `live/risk_monitor.py` 已实现

## 四个章节
1. 初始化 PaperTrader
2. 批量回放 2025 全年
3. 绩效评估
4. 风险监控回溯

In [ ]:
import sys
sys.path.insert(0, '../../..')  # 确保能 import 项目模块

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'SimHei'
matplotlib.rcParams['axes.unicode_minus'] = False

print('✅ 基础依赖加载成功')

---

## Section 1：初始化 PaperTrader

`PaperTrader` 是模拟持仓管理核心类，负责：
- 追踪虚拟持仓和现金余额
- 记录每笔交易（含手续费 0.15%）
- 计算每日净值（NAV）
- 生成绩效报告（夏普、最大回撤、年化收益等）

初始化参数：
- `initial_capital`：初始资金，默认 100 万元
- `commission_rate`：手续费率，默认 0.0015（双边 0.3%）
- `state_dir`：持仓快照保存路径，默认 `live/portfolio/`

In [ ]:
from live.paper_trader import PaperTrader

pt = PaperTrader()
print('初始资金:', pt.initial_capital)

---

## Section 2：批量回放 2025 全年模拟

对 2025 年每个月初（共 12 个交易日）执行一次 `run_daily_pipeline`，收集净值序列。

> **数据依赖说明**：本 section 需要本地 CSV 数据覆盖 2025 全年。
> 如果数据不可用，代码会打印跳过信息并继续，不会报错中断。

回放逻辑：
1. 遍历 2025 年每个月第一个交易日
2. 调用 `run_daily_pipeline` 生成选股信号
3. PaperTrader 执行虚拟交易，更新持仓
4. 记录当日净值

In [ ]:
from pipeline.daily_signal import run_daily_pipeline

# 2025 年每月月初交易日（A股节假日调整后的近似日期）
month_starts_2025 = [
    '2025-01-02', '2025-02-05', '2025-03-03',
    '2025-04-01', '2025-05-06', '2025-06-02',
    '2025-07-01', '2025-08-01', '2025-09-01',
    '2025-10-08', '2025-11-03', '2025-12-01',
]

nav_records = []  # 收集 (日期, 净值) 对

for date_str in month_starts_2025:
    try:
        # 生成当日选股信号
        signals = run_daily_pipeline(date=date_str)
        
        # PaperTrader 执行虚拟交易
        pt.execute_trades(signal_date=date_str, signals=signals)
        
        # 记录净值
        nav = pt.get_nav(date_str)
        nav_records.append({'date': date_str, 'nav': nav})
        print(f'[{date_str}] 净值: {nav:.4f}  持仓数: {len(pt.positions)}')
        
    except Exception as e:
        print(f'[{date_str}] 跳过（数据不可用）: {e}')
        continue

# 转为 DataFrame
if nav_records:
    nav_df = pd.DataFrame(nav_records).set_index('date')
    nav_df.index = pd.to_datetime(nav_df.index)
    print(f'\n✅ 回放完成，共 {len(nav_df)} 个数据点')
    print(nav_df)
else:
    print('⚠️ 无有效数据，请先运行 pipeline 生成 2025 年信号')

---

## Section 3：绩效评估

调用 `pt.get_performance()` 获取完整绩效报告，包括：
- 年化收益率
- 夏普比率（年化）
- 最大回撤
- 胜率（月度）

同时绘制净值曲线与 HS300 基准对比图。

In [ ]:
# 获取绩效报告
perf = pt.get_performance()
print('=== 模拟盘绩效报告 ===')
for k, v in perf.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

# 策略评审门槛检查
print('\n=== 策略评审门槛检查 ===')
annual_ret = perf.get('annual_return', 0)
sharpe = perf.get('sharpe_ratio', 0)
max_dd = perf.get('max_drawdown', -1)

print(f'  年化收益 > 15%:  {"✅" if annual_ret > 0.15 else "❌"} ({annual_ret:.1%})')
print(f'  夏普比率 > 0.8:  {"✅" if sharpe > 0.8 else "❌"} ({sharpe:.2f})')
print(f'  最大回撤 < 30%:  {"✅" if max_dd > -0.30 else "❌"} ({max_dd:.1%})')

In [ ]:
# 绘制净值曲线
if nav_records:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # 净值曲线
    ax1 = axes[0]
    nav_df['nav'].plot(ax=ax1, color='steelblue', linewidth=2, label='模拟盘净值')
    ax1.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='初始净值=1.0')
    ax1.set_title('2025 全年模拟盘净值曲线', fontsize=14)
    ax1.set_ylabel('净值')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # 回撤曲线
    ax2 = axes[1]
    rolling_max = nav_df['nav'].cummax()
    drawdown = (nav_df['nav'] - rolling_max) / rolling_max
    drawdown.plot(ax=ax2, color='red', linewidth=1.5, label='回撤')
    ax2.fill_between(drawdown.index, drawdown.values, 0, alpha=0.3, color='red')
    ax2.set_title('回撤曲线', fontsize=14)
    ax2.set_ylabel('回撤幅度')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ 无净值数据，跳过绘图')

---

## Section 4：风险监控回溯

调用 `check_risk_alerts` 对当前持仓状态进行全面风险扫描：

| 监控项 | 阈值 | 触发条件 |
|--------|------|----------|
| 最大回撤 | -20% | 累计净值回撤超过 20% |
| 单股集中度 | 15% | 单只股票持仓市值占比超过 15% |
| 行业集中度 | 40% | 单一行业持仓市值占比超过 40% |
| 因子 IC 衰减 | ICIR < 0.1 | 近 20 日因子 IC 均值绝对值低于 0.1 |

> **重要**：WARN 级别预警需要人工审查，ERROR 级别需要立即减仓或清仓。

In [ ]:
from live.risk_monitor import check_risk_alerts

# 执行风险扫描
alerts = check_risk_alerts(pt)

if alerts:
    print(f'=== 风险预警报告（共 {len(alerts)} 条）===')
    for alert in alerts:
        level = alert.get('level', 'INFO')
        msg = alert.get('message', '')
        value = alert.get('value', '')
        threshold = alert.get('threshold', '')
        
        emoji = {'ERROR': '🔴', 'WARN': '🟡', 'INFO': '🟢'}.get(level, '⚪')
        print(f'  {emoji} [{level}] {msg}')
        if value:
            print(f'       当前值: {value}  阈值: {threshold}')
else:
    print('✅ 无风险预警，当前持仓状态正常')

# 打印持仓分布摘要
print('\n=== 当前持仓摘要 ===')
if hasattr(pt, 'positions') and pt.positions:
    positions_df = pd.DataFrame(pt.positions).T
    print(positions_df.to_string())
else:
    print('当前无持仓（或持仓数据未加载）')